# Chapter 9 — Compact companions: prediction and follow-up

**Goal.** Distinguish a large inferred mass from a validated dark companion. Reconstruct radial-velocity
mass functions from published fits, compare them with the original Gaia predictions, and state which
assumptions survive. The small automatically downloaded extract is transcribed from identified paper tables, with units and provenance in
`data/compact_followup.json`; it contains fitted parameters, not epoch RV measurements.

Our examples are Gaia BH1 and two rejected massive-companion orbital solutions. A rejected orbit does not
mean the star is single: both rejected examples are spectroscopic binaries. We use the original 2023 BH1
fit for a like-for-like historical comparison; later ESPRESSO work refines its mass and is discussed below.
This chapter is a validation exercise, not a new search for black holes.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts, attached to a GitHub release of the tutorial. MANIFEST.json lists, for each file,
# its origin (SQL query or URL) and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/gaia-nss-tutorial/releases/download/data-v0.3/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)

## 9.1 What does the radial-velocity mass function constrain?

For a single-lined Keplerian binary,

$$f_{\rm RV}=\frac{PK_1^3(1-e^2)^{3/2}}{2\pi G}
=\frac{M_2^3\sin^3i}{(M_1+M_2)^2}.$$

A minimum companion mass follows at $\sin i=1$ once the primary mass is specified. A small mass function
cannot exclude a massive, nearly face-on companion. A large mass function still needs a reliable orbit,
a sound primary-star model and evidence that a luminous star or unresolved multiple system is absent.

Compute the function from the reported P, K and e for Gaia BH1 (El-Badry et al. 2023, Table 1,
https://arxiv.org/abs/2209.06833) and two examples from Simon et al. (2026), Table 4,
https://arxiv.org/abs/2603.20371. The BH1 RV fit fixes P=185.6 d; it is not an independent period measurement.

In [ ]:
follow = load('compact_followup')
nss = load('nss_two_body')
G, MSUN, AU, DAY = 6.67430e-11, 1.98847e30, 149597870700.0, 86400.0
f_rv = follow.period_days*DAY*(follow.k_kms*1000)**3*(1-follow.eccentricity**2)**1.5/(2*np.pi*G*MSUN)
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(follow))
ax.plot(x, f_rv, 'o', label='recomputed from P, K, e')
ax.errorbar(x+0.06, follow.f_msun, yerr=[follow.f_err_lo, follow.f_err_hi], fmt='kx',
            label='published f and marginal interval')
ax.set(xticks=x, xticklabels=follow.name, yscale='log', ylabel='RV mass function [solar masses]')
ax.legend(); plt.tight_layout(); plt.show()
# The table inputs are rounded. Allow 2% relative difference, specified before recomputing f.
assert np.allclose(f_rv, follow.f_msun, rtol=0.02, atol=0)
assert follow.source_id.is_unique and follow.source_id.dtype.kind in 'iu'
print(follow[['source_id', 'name', 'reference', 'table']].to_string(index=False))

**Figure 9.1.** Recomputed RV mass functions (blue circles) and published values with their marginal
uncertainties (black crosses) for three named systems, in solar masses on a logarithmic axis. Agreement checks
the transcription, units and formula, not the adequacy of the published orbit fit. We have not refitted RVs
or propagated a joint covariance from marginal error bars.

## 9.2 Did the Gaia orbit predict the follow-up?

Recover the photocentre semimajor axis and inclination from Thiele–Innes A, B, F, G. With
$u=(A^2+B^2+F^2+G^2)/2$ and $v=AG-BF$, $a_0^2=u+\sqrt{u^2-v^2}$ and $\cos i=v/a_0^2$.
Under the **dark-companion hypothesis**, the photocentre follows the visible star, so

$$K_{\rm predicted}=\frac{2\pi(a_0/\varpi)\sin i}{P\sqrt{1-e^2}}.$$

Convert au per year to km/s. If the companion contributes light, the photocentre is not the primary's
barycentric orbit and this prediction changes. Compare periods as well as velocity amplitude: changing
the flux ratio cannot repair a wrong orbital period. Nominal predictions below are not confidence regions.

In [ ]:
orb = nss[nss.nss_solution_type == 'Orbital']
assert orb.source_id.is_unique
comparison = follow.merge(orb, on='source_id', suffixes=('_rv', '_gaia'), validate='one_to_one')
assert len(comparison) == len(follow)
A, B, F, GG = [comparison[c+'_thiele_innes'].to_numpy(dtype=float) for c in ['a', 'b', 'f', 'g']]
u, v = (A*A+B*B+F*F+GG*GG)/2, A*GG-B*F
a0 = np.sqrt(u+np.sqrt(np.maximum(u*u-v*v, 0)))
inclination = np.arccos(np.clip(v/a0**2, -1, 1))
assert np.all(comparison.nss_parallax > 0)
a1_au = a0/comparison.nss_parallax
k_predict = 2*np.pi*a1_au*AU*np.sin(inclination)/(comparison.period*DAY*np.sqrt(1-comparison.eccentricity_gaia**2))/1000
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, gaia, measured, errlo, errhi, ylabel in [
    (ax[0], comparison.period, comparison.period_days, comparison.period_err_lo, comparison.period_err_hi, 'period [days]'),
    (ax[1], k_predict, comparison.k_kms, comparison.k_err_lo, comparison.k_err_hi, 'RV semi-amplitude [km/s]')]:
    a.plot(x-0.08, gaia, 'o', label='DR3 nominal dark-companion prediction')
    a.errorbar(x+0.08, measured, yerr=[errlo, errhi], fmt='s', color='C1', label='published RV solution')
    a.set(xticks=x, xticklabels=follow.name, yscale='log', ylabel=ylabel)
    a.tick_params(axis='x', labelrotation=12); a.legend(fontsize=6)
plt.tight_layout(); plt.show()
# Reproduce BH1's astrometric Table 1 values (same DR3 data), allowing its stated table rounding.
bh = np.flatnonzero(comparison.source_id.to_numpy() == 4373465352415301632)[0]
assert abs(a0[bh]-3.00) < 0.05 and abs(np.rad2deg(inclination[bh])-121.2) < 1.0
assert abs(comparison.period.iloc[bh]-185.77) < 0.01

**Figure 9.2.** Gaia nominal orbital predictions (blue circles) versus published RV-fit values (orange
squares): period in days (left) and velocity semi-amplitude in km/s (right), both logarithmic. RV error bars
are quoted marginal intervals; Gaia uncertainty regions are not shown. BH1's displayed RV period was fixed,
so its equality is not a validation. Its astrometric amplitude is uncertain and the nominal offset alone is
not evidence against the orbit. The two other systems have markedly different follow-up periods and amplitudes.

Simon et al. reject the massive-companion interpretations of those two Gaia orbits while confirming that the
stars are binaries. Their follow-up tests the orbit, not merely the presence of variability. A different RV
period in an arbitrary system could instead trace an inner binary, as Chapter 8 illustrates; use the full
source-level evidence before deciding between those alternatives.

## 9.3 Inclination and primary mass still matter

Solve the mass-function equation for a range of inclinations using a bracketing root finder. For BH1 adopt
the original study's primary mass 0.93 solar masses. Use 1 solar mass for the other examples **only as an
illustrative assumption**; those curves are not new companion-mass measurements. The minimum at 90 degrees
is a conditional lower bound, while smaller inclinations allow larger masses.

In [ ]:
from scipy.optimize import brentq

def secondary_mass(f, primary, sin_i):
    # The function is monotonic for positive companion mass; bracket a positive root.
    return brentq(lambda m: m**3*sin_i**3/(primary+m)**2-f, 1e-8, 1e5, xtol=1e-12)

angles = np.linspace(15, 90, 200)
primary = [0.93, 1.0, 1.0]
fig, ax = plt.subplots(figsize=(8, 4))
for j, name in enumerate(follow.name):
    mass = [secondary_mass(f_rv.iloc[j], primary[j], np.sin(np.deg2rad(i))) for i in angles]
    ax.plot(angles, mass, label=f'{name}; assumed M1={primary[j]}')
ax.set(xlabel='assumed inclination [degrees]', ylabel='companion mass [solar masses]', yscale='log')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
# Independent injected binary: recover its known secondary, including a non-edge-on orbit.
known_m1, known_m2, known_i = 1.1, 0.7, np.deg2rad(43)
known_f = known_m2**3*np.sin(known_i)**3/(known_m1+known_m2)**2
assert abs(secondary_mass(known_f, known_m1, np.sin(known_i))-known_m2) < 1e-10
assert secondary_mass(known_f, known_m1, 1.0) < known_m2

**Figure 9.3.** Companion mass versus assumed inclination, in degrees and solar masses, with a logarithmic
mass axis. Blue uses BH1's RV mass function and its published primary mass; orange and green use the other
RV functions but assume a one-solar-mass primary for illustration. All curves increase towards face-on orbits.
A low RV mass function is therefore not, by itself, a proof that the companion is low mass.

## 9.4 Keep the likelihood and the evidence attached to a mass

BH1's original astrometry-only mass, original joint astrometry+RV mass, and later ESPRESSO-based mass answer
related questions with different inputs. Plot the published summaries without treating them as independent
measurements: they reuse Gaia astrometry. The later study tests an inner-binary interpretation and reports
no evidence for it (Nagarajan et al. 2024, https://arxiv.org/abs/2312.05313). The values below are literature
summaries, not three results independently rederived by this notebook.

In [ ]:
labels = ['2023: Gaia astrometry only', '2023: Gaia + RVs', '2024: Gaia + extended RVs']
values, errors = np.array([12.8, 9.62, 9.27]), np.array([2.0, 0.18, 0.10])
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.errorbar(values, np.arange(3), xerr=errors, fmt='o', color='C0', capsize=3)
ax.set(yticks=np.arange(3), yticklabels=labels, xlabel='published Gaia BH1 companion mass [solar masses]')
ax.invert_yaxis(); plt.tight_layout(); plt.show()
# Numerical reading check: the original joint value lies within the original astrometric 2-sigma interval.
# This is not an independent-data consistency test: the two solutions share Gaia measurements.
assert abs(values[1]-values[0]) < 2*errors[0]

**Figure 9.4.** Published BH1 mass summaries: original astrometry only, original joint fit, and the later
extended-RV analysis. Horizontal bars are reported one-standard-error intervals in solar masses. Their
progressive precision reflects additional information and changed fits, not repeated independent detections.
The final point is the cited 2024 result, not a claim to have surveyed every later determination.

A defensible report names the catalogue release, solution type, data actually fitted, assumptions about light
and primary mass, and the follow-up outcome. Keep a candidate label when those checks remain open. Failed
predictions are useful: they expose aliases, blends or incorrect models before a population claim is made.

**Capstone.** Choose a separate `Orbital`+`SB1` source from Chapter 8. Assemble its period, eccentricity,
photocentre orbit, parallax, quality flags and flux information; predict its RV amplitude under a dark-companion
hypothesis. Describe which new observations would discriminate a dark secondary, a luminous companion and a
hierarchical system. Do not convert the candidate count into an occurrence rate without a selection model.

The core tutorial now reaches candidate selection, orbit fitting, cross-channel checks and follow-up reasoning.
The separate selection-function extension (Chapter 10) begins population inference; wide binaries and a full
public DR4 catalogue analysis remain future topics.

## References

- El-Badry et al. (2023), *A Sun-like star orbiting a black hole*, MNRAS 518, 1057, Table 1: https://arxiv.org/abs/2209.06833 ; https://doi.org/10.1093/mnras/stac3140
- Simon et al. (2026), *Radial Velocity Orbital Solutions for Candidate Black Hole and Neutron Star Binary Systems in the Gaia Data Release 3 Catalog*, ApJ 1007, 160, Table 4: https://arxiv.org/abs/2603.20371 ; https://doi.org/10.3847/1538-4357/ae7e8a
- Nagarajan et al. (2024), *ESPRESSO Observations of Gaia BH1: High-precision Orbital Constraints and no Evidence for an Inner Binary*, PASP 136, 014202: https://arxiv.org/abs/2312.05313 ; https://doi.org/10.1088/1538-3873/ad1ba7